# Boltz-2 ONNX fp16: inspect and view the graphs

This notebook downloads the public fp16 ONNX export, checks the graph signatures, detects the available runtime providers, and opens the model in Netron. The exported graphs require the Boltz preprocessing/orchestration pipeline for end-to-end protein prediction; this notebook focuses on graph inspection and runtime validation.

## 1. Install the notebook dependencies

In Colab, select **Runtime → Change runtime type → T4 GPU** when a GPU is available. The notebook also works on CPU.

In [ ]:
!pip -q install onnx onnxruntime-gpu netron huggingface_hub graphviz pydot

import os
import sys
import subprocess
from pathlib import Path

MODEL_DIR = Path('/content/boltz-2-onnx-fp16')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version.split()[0])
print('Working directory:', MODEL_DIR)

## 2. Download the fp16 model

The weights are fetched from the public Hugging Face repository. External-data sidecars (`*.onnx.data`) are downloaded alongside each ONNX graph.

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id='latentspacecraft/boltz-2-onnx',
    repo_type='model',
    local_dir=str(MODEL_DIR),
    allow_patterns=['fp16/*', 'README.md', 'meta.json'],
)

for path in sorted((MODEL_DIR / 'fp16').iterdir()):
    print(f'{path.name:32} {path.stat().st_size / 1024**2:8.1f} MB')

## 3. Inspect graph metadata and runtime providers

In [ ]:
import onnx
import onnxruntime as ort

print('ONNX Runtime:', ort.__version__)
print('Available execution providers:', ort.get_available_providers())

graph_paths = {
    'trunk': MODEL_DIR / 'fp16' / 'trunk_fp16.onnx',
    'diffusion_step': MODEL_DIR / 'fp16' / 'diffusion_step_fp16.onnx',
    'confidence': MODEL_DIR / 'fp16' / 'confidence_fp16.onnx',
}

def tensor_shape(value_info):
    dims = []
    for dim in value_info.type.tensor_type.shape.dim:
        dims.append(dim.dim_param or dim.dim_value or '?')
    return dims

for name, path in graph_paths.items():
    model = onnx.load(str(path), load_external_data=False)
    print(f'\n{name}: {path.name}')
    print(f'  opset: {model.opset_import[0].version}')
    print(f'  nodes: {len(model.graph.node):,}')
    print(f'  inputs: {len(model.graph.input):,}  outputs: {len(model.graph.output):,}')
    print('  first inputs:')
    for value in model.graph.input[:5]:
        print(f'    - {value.name}: {tensor_shape(value)}')
    print('  outputs:')
    for value in model.graph.output:
        print(f'    - {value.name}: {tensor_shape(value)}')

## 4. Open a bounded graph preview in Netron

The complete trunk contains too many nodes for a browser tab to render reliably. Opening the full graph can crash Chrome/Colab with an **Out of Memory** error. The next cell creates a small structural preview from the first nodes and opens only that preview in Netron. The complete graph remains available for the metadata and provider checks above.

In [ ]:
import copy
import netron
from IPython.display import HTML, display
from google.colab import output

PREVIEW_NODES = 250
full_model = onnx.load(str(graph_paths['trunk']), load_external_data=False)
preview_nodes = [copy.deepcopy(node) for node in full_model.graph.node[:PREVIEW_NODES]]
preview_outputs = []
for node in preview_nodes[-10:]:
    for output_name in node.output:
        if output_name:
            preview_outputs.append(onnx.helper.make_tensor_value_info(
                output_name, onnx.TensorProto.FLOAT, None))

preview_graph = onnx.helper.make_graph(
    preview_nodes,
    'trunk_structural_preview',
    [copy.deepcopy(value) for value in full_model.graph.input],
    preview_outputs,
)
preview_model = onnx.helper.make_model(
    preview_graph,
    producer_name='boltz-2-onnx-colab-preview',
    opset_imports=[copy.deepcopy(opset) for opset in full_model.opset_import],
)
preview_path = MODEL_DIR / 'trunk_structural_preview.onnx'
onnx.save(preview_model, str(preview_path))
print(f'Created {preview_path.name} with {len(preview_nodes):,} of {len(full_model.graph.node):,} nodes.')
print('This is a visualization preview, not an executable inference model.')

NETRON_PORT = 8081
netron.start(str(preview_path), address='0.0.0.0', port=NETRON_PORT, browse=False)
public_url = output.serve_kernel_port_as_window(NETRON_PORT, anchor_text='Open bounded trunk preview in Netron')
display(HTML(f'<p><a href="{public_url}" target="_blank">Open bounded trunk preview in Netron</a></p>'))
print('Do not open the original full trunk in Netron; it can exhaust browser memory.')

## 5. Optional: verify the GPU execution provider

This creates an ONNX Runtime session without running inference. Full inference needs correctly shaped feature tensors from the Boltz input pipeline.

In [ ]:
providers = ort.get_available_providers()
preferred = ['CUDAExecutionProvider', 'CPUExecutionProvider']
session_providers = [p for p in preferred if p in providers]
if not session_providers:
    raise RuntimeError(f'No supported ONNX Runtime provider found. Available: {providers}')

session = ort.InferenceSession(str(graph_paths['trunk']), providers=session_providers)
print('Session providers:', session.get_providers())
print('Graph inputs:', len(session.get_inputs()))
print('Graph outputs:', len(session.get_outputs()))
if 'CUDAExecutionProvider' in session.get_providers():
    print('GPU execution is available.')
else:
    print('CUDA is unavailable; this session is using CPU.')